# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsiddan25/program/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: Growing content is longer and younger

The paper observed that growing pages were 37.6% longer and 20% younger than declining pages. The label comes from `trend_direction`, based on the impression change between the latest 30 days and the previous 30 days. Word count and age are comparison variables, not labels.

Methodology question: Were pages compared within similar clients, content types, and search-demand levels? The results show an association, but they do not prove that making a page longer causes growth.

Finding 2: Content peaks at 61–90 days

The paper observed the highest average health score among pages aged 61–90 days and lower performance after 270 days. There is no binary label here: content age defines the groups, while FlyRank’s composite health score is the measured outcome.

Methodology question: Were the same pages followed over time, or were different age groups compared at one snapshot? The design supports an observed age-related pattern, but not the claim that every page naturally peaks and declines at these ages.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)


Paste your Hugging Face READ token (hf_...): ··········


In [2]:
import os
import sys
import duckdb
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.13.15
pandas: 2.2.3
NumPy: 2.1.3
scikit-learn: 1.6.1


In [3]:
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_content": (
        f"read_parquet('{REL}/dim_content.parquet')"
    ),
    "fact_daily": (
        f"read_parquet("
        f"'{REL}/fact_content_daily_performance/**/*.parquet'"
        f")"
    ),
}

print("DuckDB connection and table paths are ready.")

DuckDB connection and table paths are ready.


In [4]:
MONTHS = [
    "2026-02",
    "2026-03",
    "2026-04",
    "2026-05",
    "2026-06",
]

monthly_queries = []

for month in MONTHS:
    monthly_queries.append(f"""
        SELECT
            client_hash_id,
            content_hash_id,
            '{month}' AS month_key,

            COUNT(*) AS observed_days,

            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_sum_position) AS sum_position,

            AVG(gsc_impressions) AS daily_impression_mean,
            STDDEV_SAMP(gsc_impressions) AS daily_impression_std,

            STDDEV_SAMP(
                CASE
                    WHEN gsc_impressions > 0
                    THEN gsc_avg_position
                END
            ) AS daily_position_std,

            SUM(
                CASE
                    WHEN gsc_impressions > 0 THEN 1
                    ELSE 0
                END
            ) AS days_with_impressions

        FROM read_parquet(
            '{REL}/fact_content_daily_performance/'
            'month={month}/*.parquet'
        )

        WHERE gsc_data_available IS TRUE

        GROUP BY
            client_hash_id,
            content_hash_id
    """)

monthly_union_sql = "\nUNION ALL\n".join(monthly_queries)

monthly_summary = con.sql(monthly_union_sql).df()

monthly_summary = monthly_summary.sort_values(
    ["month_key", "client_hash_id", "content_hash_id"]
).reset_index(drop=True)

print(f"Monthly summary rows: {len(monthly_summary):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Monthly summary rows: 971,603


In [5]:
monthly_coverage = (
    monthly_summary
    .groupby("month_key")
    .agg(
        page_rows=("content_hash_id", "size"),
        clients=("client_hash_id", "nunique"),
        minimum_days=("observed_days", "min"),
        maximum_days=("observed_days", "max"),
    )
    .reset_index()
)

monthly_coverage

,month_key,page_rows,clients,minimum_days,maximum_days
0,2026-02,153559,46,1,28
1,2026-03,176738,47,1,31
2,2026-04,194760,51,1,30
3,2026-05,237910,56,1,31
4,2026-06,208636,55,1,30


In [6]:
monthly_cache_path = "/content/flyrank_monthly_summary.pkl"

monthly_summary.to_pickle(monthly_cache_path)

print(f"Temporary cache saved: {monthly_cache_path}")

Temporary cache saved: /content/flyrank_monthly_summary.pkl


In [7]:
monthly_value_columns = [
    "observed_days",
    "impressions",
    "clicks",
    "sum_position",
    "daily_impression_mean",
    "daily_impression_std",
    "daily_position_std",
    "days_with_impressions",
]


def select_month(month, prefix):
    month_frame = monthly_summary.loc[
        monthly_summary["month_key"] == month,
        [
            "client_hash_id",
            "content_hash_id",
            *monthly_value_columns,
        ],
    ].copy()

    rename_map = {
        column: f"{prefix}_{column}"
        for column in monthly_value_columns
    }

    return month_frame.rename(columns=rename_map)


february = select_month("2026-02", "feb")
march = select_month("2026-03", "mar")
april = select_month("2026-04", "apr")
may = select_month("2026-05", "may")

In [8]:
development = (
    february
    .merge(
        march,
        on=["client_hash_id", "content_hash_id"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        april,
        on=["client_hash_id", "content_hash_id"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        may,
        on=["client_hash_id", "content_hash_id"],
        how="inner",
        validate="one_to_one",
    )
)

development = development[
    (development["feb_observed_days"] >= 20)
    & (development["mar_observed_days"] >= 20)
    & (development["apr_observed_days"] >= 20)
    & (development["may_observed_days"] >= 20)
    & (development["mar_impressions"] > 0)
    & (development["apr_impressions"] > 0)
].copy()

development = development.reset_index(drop=True)

print(f"Development rows: {len(development):,}")
print(
    "Development clients:",
    development["client_hash_id"].nunique(),
)
print(
    "Duplicate client-page rows:",
    development.duplicated(
        ["client_hash_id", "content_hash_id"]
    ).sum(),
)

Development rows: 62,558
Development clients: 23
Duplicate client-page rows: 0


In [9]:
# Three-month traffic totals: February through April
development["impressions_90d"] = (
    development["feb_impressions"]
    + development["mar_impressions"]
    + development["apr_impressions"]
)

development["clicks_90d"] = (
    development["feb_clicks"]
    + development["mar_clicks"]
    + development["apr_clicks"]
)

# Log versions reduce the effect of extremely large traffic values.
development["log_impressions_90d"] = np.log1p(
    development["impressions_90d"]
)

development["log_previous_month_impressions"] = np.log1p(
    development["mar_impressions"]
)

development["log_current_month_impressions"] = np.log1p(
    development["apr_impressions"]
)

# March-to-April impression momentum
development["recent_change_pct"] = (
    100
    * (
        development["apr_impressions"]
        - development["mar_impressions"]
    )
    / development["mar_impressions"]
)

# Limit extreme growth percentages caused by small denominators.
development["recent_change_pct_clipped"] = (
    development["recent_change_pct"]
    .clip(lower=-100, upper=500)
)

# Monthly CTR
development["previous_ctr"] = (
    100
    * development["mar_clicks"]
    / development["mar_impressions"]
)

development["current_ctr"] = (
    100
    * development["apr_clicks"]
    / development["apr_impressions"]
)

development["ctr_change"] = (
    development["current_ctr"]
    - development["previous_ctr"]
)

# Impression-weighted average position
development["previous_avg_position"] = (
    development["mar_sum_position"]
    / development["mar_impressions"]
)

development["current_avg_position"] = (
    development["apr_sum_position"]
    / development["apr_impressions"]
)

# Positive means the page's average position became worse.
development["position_change"] = (
    development["current_avg_position"]
    - development["previous_avg_position"]
)

# Current-month traffic volatility
development["current_impression_cv"] = (
    development["apr_daily_impression_std"]
    / development["apr_daily_impression_mean"]
)

development["current_position_std"] = (
    development["apr_daily_position_std"]
)

development["current_impression_day_rate"] = (
    development["apr_days_with_impressions"]
    / development["apr_observed_days"]
)

# May outcome: validation only, never a feature
development["future_change_pct"] = (
    100
    * (
        development["may_impressions"]
        - development["apr_impressions"]
    )
    / development["apr_impressions"]
)

development["future_decline"] = (
    development["may_impressions"]
    < 0.80 * development["apr_impressions"]
).astype(int)

In [10]:
print(
    "Future-decline base rate:",
    f"{development['future_decline'].mean():.2%}",
)

print(
    "Infinite feature values:",
    np.isinf(
        development.select_dtypes(include="number")
    ).sum().sum(),
)

print(
    "Missing current position volatility:",
    development["current_position_std"].isna().sum(),
)

Future-decline base rate: 48.79%
Infinite feature values: 0
Missing current position volatility: 0


In [11]:
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold,
)

In [13]:
# ============================================================
# SECTION 2: RANDOM VERSUS CLIENT-GROUPED VALIDATION
# ============================================================

feature_columns = [
    # Traffic level
    "log_impressions_90d",
    "log_previous_month_impressions",
    "log_current_month_impressions",

    # Recent traffic direction
    "recent_change_pct_clipped",

    # CTR level and movement
    "previous_ctr",
    "current_ctr",
    "ctr_change",

    # Search position level and movement
    "previous_avg_position",
    "current_avg_position",
    "position_change",

    # Current-month stability
    "current_impression_cv",
    "current_position_std",
    "current_impression_day_rate",
]



X = (
    development[feature_columns]
    .reset_index(drop=True)
    .copy()
)

y = (
    development["future_decline"]
    .astype(int)
    .reset_index(drop=True)
)

groups = (
    development["client_hash_id"]
    .astype(str)
    .reset_index(drop=True)
)

print(f"Evaluation rows: {len(X):,}")
print(f"Clients: {groups.nunique()}")
print(f"Future-decline base rate: {y.mean():.2%}")
print(f"Features: {len(feature_columns)}")

Evaluation rows: 62,558
Clients: 23
Future-decline base rate: 48.79%
Features: 13


In [14]:
def build_random_forest():
    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=200,
                    max_depth=10,
                    min_samples_leaf=25,
                    class_weight="balanced_subsample",
                    n_jobs=-1,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

In [15]:
def generate_oof_predictions(
    splitter,
    X,
    y,
    groups,
    design_name,
    use_groups,
):
    # Every page will eventually receive one validation prediction.
    oof_scores = np.full(len(X), np.nan)

    fold_records = []

    if use_groups:
        split_iterator = splitter.split(
            X,
            y,
            groups=groups,
        )
    else:
        split_iterator = splitter.split(X, y)

    for fold_number, (train_index, validation_index) in enumerate(
        split_iterator,
        start=1,
    ):
        fold_model = build_random_forest()

        X_fold_train = X.iloc[train_index]
        y_fold_train = y.iloc[train_index]

        X_fold_validation = X.iloc[validation_index]
        y_fold_validation = y.iloc[validation_index]

        fold_model.fit(
            X_fold_train,
            y_fold_train,
        )

        oof_scores[validation_index] = (
            fold_model.predict_proba(
                X_fold_validation
            )[:, 1]
        )

        train_clients = set(groups.iloc[train_index])
        validation_clients = set(groups.iloc[validation_index])

        fold_records.append({
            "validation_design": design_name,
            "fold": fold_number,
            "training_rows": len(train_index),
            "validation_rows": len(validation_index),
            "training_clients": len(train_clients),
            "validation_clients": len(validation_clients),
            "overlapping_clients": len(
                train_clients.intersection(validation_clients)
            ),
            "validation_base_rate": (
                y_fold_validation.mean()
            ),
        })

    # Every row should have exactly one out-of-fold score.
    assert not np.isnan(oof_scores).any()

    return oof_scores, pd.DataFrame(fold_records)

In [16]:
random_splitter = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

random_oof_scores, random_fold_summary = (
    generate_oof_predictions(
        splitter=random_splitter,
        X=X,
        y=y,
        groups=groups,
        design_name="Random row CV — before",
        use_groups=False,
    )
)

In [17]:
grouped_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

grouped_oof_scores, grouped_fold_summary = (
    generate_oof_predictions(
        splitter=grouped_splitter,
        X=X,
        y=y,
        groups=groups,
        design_name="Client-grouped CV — after",
        use_groups=True,
    )
)

In [18]:
fold_summary = pd.concat(
    [
        random_fold_summary,
        grouped_fold_summary,
    ],
    ignore_index=True,
)

fold_summary["validation_base_rate"] = (
    100 * fold_summary["validation_base_rate"]
).round(2)

display(fold_summary)

,validation_design,fold,training_rows,validation_rows,training_clients,validation_clients,overlapping_clients,validation_base_rate
0,Random row CV — before,1,50046,12512,23,21,21,48.79
1,Random row CV — before,2,50046,12512,23,22,22,48.79
2,Random row CV — before,3,50046,12512,23,23,23,48.79
3,Random row CV — before,4,50047,12511,23,23,23,48.79
4,Random row CV — before,5,50047,12511,23,22,22,48.79
5,Client-grouped CV — after,1,62257,301,22,1,0,32.89
6,Client-grouped CV — after,2,56197,6361,16,7,0,70.98
7,Client-grouped CV — after,3,58695,3863,22,1,0,47.89
8,Client-grouped CV — after,4,40733,21825,18,5,0,54.81
9,Client-grouped CV — after,5,32350,30208,14,9,0,40.04


In [19]:
grouped_overlap = grouped_fold_summary[
    "overlapping_clients"
].sum()

assert grouped_overlap == 0

print("PASS: Every grouped fold has zero client overlap.")

PASS: Every grouped fold has zero client overlap.


In [20]:
def precision_at_k(y_true, scores, k):
    y_array = np.asarray(y_true)
    score_array = np.asarray(scores)

    ranked_indices = np.argsort(-score_array)
    selected_indices = ranked_indices[:min(k, len(ranked_indices))]

    return y_array[selected_indices].mean()


def evaluate_scores(
    validation_design,
    method,
    y_true,
    scores,
):
    y_array = np.asarray(y_true)
    score_array = np.asarray(scores)

    return {
        "validation_design": validation_design,
        "method": method,
        "rows_evaluated": len(y_array),
        "base_rate": y_array.mean(),
        "precision_at_20": precision_at_k(
            y_array, score_array, 20
        ),
        "precision_at_50": precision_at_k(
            y_array, score_array, 50
        ),
        "precision_at_100": precision_at_k(
            y_array, score_array, 100
        ),
        "average_precision": average_precision_score(
            y_array, score_array
        ),
        "roc_auc": roc_auc_score(
            y_array, score_array
        ),
    }

In [23]:
baseline_qualifies = (
    (development["mar_impressions"] >= 300)
    & (development["recent_change_pct"] < -20)
)

baseline_scores = np.where(
    baseline_qualifies,
    (
        development["mar_impressions"]
        - development["apr_impressions"]
    ),
    0,
)

In [24]:
validation_comparison = pd.DataFrame([
    evaluate_scores(
        validation_design="Reference rule",
        method="Week-4 baseline",
        y_true=y,
        scores=baseline_scores,
    ),
    evaluate_scores(
        validation_design="Random row CV — before",
        method="Random Forest",
        y_true=y,
        scores=random_oof_scores,
    ),
    evaluate_scores(
        validation_design="Client-grouped CV — after",
        method="Random Forest",
        y_true=y,
        scores=grouped_oof_scores,
    ),
])

percentage_columns = [
    "base_rate",
    "precision_at_20",
    "precision_at_50",
    "precision_at_100",
    "average_precision",
    "roc_auc",
]

validation_comparison[percentage_columns] = (
    100
    * validation_comparison[percentage_columns]
).round(2)

display(validation_comparison)

,validation_design,method,rows_evaluated,base_rate,precision_at_20,precision_at_50,precision_at_100,average_precision,roc_auc
0,Reference rule,Week-4 baseline,62558,48.79,75.0,66.0,73.0,56.13,59.71
1,Random row CV — before,Random Forest,62558,48.79,100.0,100.0,97.0,78.88,78.62
2,Client-grouped CV — after,Random Forest,62558,48.79,95.0,96.0,98.0,75.62,75.31


Random row-level cross-validation produced client overlap in every fold: between 21 and 23 clients appeared in both training and validation. Client-grouped cross-validation reduced this overlap to zero in all five folds, so every grouped out-of-fold prediction came from a model that had not trained on that client.

The Random Forest’s average precision decreased from 78.88% under random validation to 75.62% under client-grouped validation, while ROC AUC decreased from 78.62% to 75.31%. This observed reduction suggests that random row validation produced a mildly optimistic estimate by allowing the model to benefit from within-client similarities.

Under client-grouped validation, the Random Forest measured 95% Precision@20, 96% Precision@50, and 98% Precision@100. On the same 62,558-page population, the Week-4 baseline measured 75%, 66%, and 73%, respectively. The grouped Random Forest also exceeded the baseline in average precision and ROC AUC. These results support using the model as a decision-support ranking tool for eligible pages, but they do not establish equal performance for every client.

Grouped fold sizes and base rates varied substantially because clients contributed very different numbers of pages. The reported aggregate metrics remain page-weighted and may be influenced more heavily by large clients. In addition, evaluation eligibility required sufficient May GSC coverage, so the results should not automatically be generalized to pages without comparable outcome coverage.



## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# SECTION 3: LEAKAGE AND POPULATION AUDIT
# ============================================================

model_features = [str(column) for column in X.columns]

print("Features supplied to the model:")

for number, feature in enumerate(model_features, start=1):
    print(f"{number:>2}. {feature}")

    # Columns that must not be model inputs
forbidden_exact_names = {
    # May outcome information
    "may_impressions",
    "may_clicks",
    "may_sum_position",
    "may_observed_days",
    "future_change_pct",
    "future_decline",
    "target",
    "label",

    # Identifiers
    "client_hash_id",
    "content_hash_id",
    "keyword_hash_id",
    "url_hash_id",

    # Existing FlyRank decisions
    "health_score",
    "priority_score",
    "action_type",
    "action_label",
    "reason_code",
    "refresh_flag",

    # Post-cutoff information
    "updated_after_feature_window",
}

forbidden_name_patterns = (
    "may_",
    "_may",
    "future_",
    "_future",
    "target_",
    "_target",
    "label_",
    "_label",
    "_flag",
    "flag_",
)

normalized_features = {
    feature: feature.lower().strip()
    for feature in model_features
}

exact_name_leaks = [
    feature
    for feature, normalized in normalized_features.items()
    if normalized in forbidden_exact_names
]

pattern_leaks = [
    feature
    for feature, normalized in normalized_features.items()
    if any(
        pattern in normalized
        for pattern in forbidden_name_patterns
    )
]

suspected_leaks = sorted(
    set(exact_name_leaks + pattern_leaks)
)

Features supplied to the model:
 1. log_impressions_90d
 2. log_previous_month_impressions
 3. log_current_month_impressions
 4. recent_change_pct_clipped
 5. previous_ctr
 6. current_ctr
 7. ctr_change
 8. previous_avg_position
 9. current_avg_position
10. position_change
11. current_impression_cv
12. current_position_std
13. current_impression_day_rate


In [26]:
target_copy_columns = []

target_values = (
    pd.Series(y)
    .reset_index(drop=True)
)

for feature in model_features:
    feature_values = (
        pd.to_numeric(
            X[feature],
            errors="coerce",
        )
        .reset_index(drop=True)
    )

    comparable = (
        feature_values.notna()
        & target_values.notna()
    )

    if (
        comparable.any()
        and feature_values[comparable].equals(
            target_values[comparable].astype(
                feature_values.dtype
            )
        )
    ):
        target_copy_columns.append(feature)

In [27]:
grouped_overlap_total = int(
    grouped_fold_summary[
        "overlapping_clients"
    ].sum()
)

grouped_folds_with_overlap = int(
    (
        grouped_fold_summary[
            "overlapping_clients"
        ] > 0
    ).sum()
)

In [28]:
leakage_audit = {
    "feature_window": (
        "2026-02-01 through 2026-04-30"
    ),
    "prediction_time": "End of April 2026",
    "outcome_definition": (
        "May impressions are more than 20% "
        "below April impressions"
    ),
    "number_of_model_features": len(model_features),
    "forbidden_feature_names_found": suspected_leaks,
    "target_copy_columns_found": target_copy_columns,
    "grouped_folds_with_client_overlap": (
        grouped_folds_with_overlap
    ),
    "total_grouped_client_overlaps": (
        grouped_overlap_total
    ),
}

print("\nLeakage audit:")

for check, result in leakage_audit.items():
    print(f"{check}: {result}")


Leakage audit:
feature_window: 2026-02-01 through 2026-04-30
prediction_time: End of April 2026
outcome_definition: May impressions are more than 20% below April impressions
number_of_model_features: 13
forbidden_feature_names_found: []
target_copy_columns_found: []
grouped_folds_with_client_overlap: 0
total_grouped_client_overlaps: 0


In [29]:
assert not suspected_leaks, (
    "Possible future, label, identifier, or "
    f"product leakage found: {suspected_leaks}"
)

assert not target_copy_columns, (
    "Features reproducing the target were found: "
    f"{target_copy_columns}"
)

assert grouped_overlap_total == 0, (
    "At least one grouped fold contains "
    "client overlap."
)

assert len(X) == len(y) == len(groups)

print("\nPASS: No structural feature leakage was detected.")
print("PASS: Every grouped fold has zero client overlap.")


PASS: No structural feature leakage was detected.
PASS: Every grouped fold has zero client overlap.


In [30]:
print("\nPOPULATION-SELECTION DISCLOSURE:")

print(
    "The evaluation population requires at least "
    "20 observed GSC days in May."
)

print(
    "May values are not model features, but May "
    "availability is used to determine which pages "
    "have a sufficiently reliable outcome."
)

print(
    "Therefore, results apply to pages with adequate "
    "May GSC coverage, not necessarily to every page "
    "in the portfolio."
)


POPULATION-SELECTION DISCLOSURE:
The evaluation population requires at least 20 observed GSC days in May.
May values are not model features, but May availability is used to determine which pages have a sufficiently reliable outcome.
Therefore, results apply to pages with adequate May GSC coverage, not necessarily to every page in the portfolio.


Feature leakage audit: PASS. No May performance, target-derived variables, product-generated decisions, or identifiers were included among the 13 model features. Every grouped validation fold had zero client overlap. February through April information was used as model input, while May impressions were used to construct the future-decline outcome.

Population-selection limitation: Inclusion required at least 20 observed GSC days in May. Therefore, outcome-window availability influenced which pages were evaluated, although May performance did not enter the model. The measured results apply to pages with sufficient May GSC coverage and should not automatically be generalized to pages without comparable coverage.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

It was observed that the grouped Random Forest model is more decision oriented than the other model, which used random rows to train and validate and hence there were multiple overlapping clients. While the other model seemed to perform better in terms of the metrics, it can be observed that it is midly optimistic: it is benefitting from the training data of the client and is able to make better decision on the validation data for that same client. The grouped Random model is a better choice since it measures approximately the same as the other model and is able to validate on clients that it has not seen, which is a very strong quality.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.